# Paper15 calendar-month labels and idealized monthly returns

Four fixed-setting RF/XGBoost × 5/10-year rolling arms, annual refits, monthly
predictions and matching past OHLC eligibility. The 15 verified signal inputs
are reused unchanged. The target is exact adjusted month-end close-to-close
return, then within-month deciles. RET uses matching 60/120 completed months.

The backtest assumes that each selected stock earns that monthly return from
the signal close. This is an **idealized same-close assumption**, not an
executable daily-fill test. No intramonth stops are inferred. Costs use drifted
monthly notionals. Missing selected returns withhold the whole continuous path.

Primary: January 2020–December 2023 (48 monthly returns). Matched calendar:
February 2020–December 2023 (47 returns), separately initialized flat. Risk
metrics use monthly returns; drawdowns only measure month-end observations.

[Method and limitations](../docs/PAPER15_MONTHLY_BASELINE_20261026.md).
All cells below authenticate saved independent proofs and read small reports.
**Run All never prepares data, loads models, fits or launches a backtest.**
The original executable-return baseline and master notebook remain unchanged.


In [ ]:
from pathlib import Path
import json
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/dmv2").is_dir())
sys.path.insert(0, str(ROOT / "src"))
config_path = ROOT / "configs/paper15_monthly.local.json"
private = json.loads(config_path.read_text()) if config_path.exists() else {}
MONTHLY_RUN = Path(private.get("output_directory", ROOT / "outputs/paper15_monthly_baseline/20261026-v1"))
print("Workspace:", ROOT)
print("Saved run:", MONTHLY_RUN)


In [ ]:
from dmv2.paper15_monthly_reporting import read_paper15_monthly
monthly_saved = None
try:
    monthly_saved = read_paper15_monthly(MONTHLY_RUN)
except FileNotFoundError:
    print("Private saved evidence unavailable. No computation started.")
else:
    print(monthly_saved["status"], monthly_saved["fits_completed"], "/ 16 fits")
    print("Active phase:", monthly_saved["phase"])
    if monthly_saved["status"] == "COMPLETE_VERIFIED":
        print(monthly_saved["full_period"])
        print(monthly_saved["matched_period"])
        print("Monthly paths:", monthly_saved["complete_paths"], "complete;", monthly_saved["withheld_paths"], "withheld")
        print("No stops; gross 0 bp / net 10 bp one-way costs; month-end risk metrics")
        if "common_primary" in monthly_saved:
            print("COMMON COVERAGE DIAGNOSTIC:", monthly_saved["common_period"])
            display(monthly_saved["common_primary"])
            display(monthly_saved["common_comparison"])
            display(monthly_saved["common_annual"])
        print("FULL-PERIOD RESULTS: missing values remain withheld")
        display(monthly_saved["primary"])
        display(monthly_saved["prediction_metrics"])
        display(monthly_saved["class_performance"])
        display(monthly_saved["coverage"])
        display(monthly_saved["gaps"])
        display(monthly_saved["comparison"])
        display(monthly_saved["annual"])
    else:
        print("Independent completion pending; no intermediate scores are used to change settings.")
